# Argentine Spanish ASR full evaluation
Resumable, parallel transcription plus raw and normalized WER/CER, breakdowns, and audio inspection. Set `SAMPLE_LIMIT = 20` for a smoke run.

## Dataset

This evaluation uses the [Crowdsourced High-Quality Argentine Spanish Speech Dataset](https://www.openslr.org/61/) from OpenSLR (SLR61). It contains nearly 6,000 short audio recordings from multiple speakers, paired with reference transcriptions in Argentine Spanish. The clips average approximately five seconds and include male, female, and weather-related subsets.

The dataset and its accompanying resources can be downloaded from the [OpenSLR SLR61 page](https://www.openslr.org/61/).

In [ ]:
import json
import os
import threading
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import jiwer
import numpy as np
import pandas as pd
import requests
from IPython.display import Audio, display
from tqdm.auto import tqdm

In [ ]:
DATA_DIR = Path("/resources/data/asr/es_ar_speech")
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8999")
API_TRANSCRIBE_ENDPOINT = f"{API_BASE_URL}/asr/transcribe"
MAX_WORKERS = 4
TIMEOUT = (10, 180)
MAX_RETRIES = 3
SAMPLE_LIMIT = None
CHECKPOINT = Path("results/es_ar_asr_transcriptions.csv")
RESULT_COLUMNS = [
    "file",
    "hypothesis",
    "response_json",
    "status",
    "error",
    "attempts",
    "elapsed_seconds",
]

## Load and validate annotations

In [ ]:
parts = []
for path in sorted(DATA_DIR.glob("*.tsv")):
    part = pd.read_csv(path, sep="\t", header=None, names=["file", "transcription"])
    part["source_tsv"] = path.name
    parts.append(part)
df = pd.concat(parts, ignore_index=True)
df.info()

In [ ]:
df.sample(5)

In [ ]:
df.loc[df["file"].duplicated(keep=False)].sort_values("file")

In [ ]:
df.loc[df.duplicated(subset=["file", "transcription"], keep=False)].sort_values("file")

In [ ]:
# Remove duplicates with same file and transcription, keeping the first occurrence
df.drop_duplicates(subset=["file", "transcription"], inplace=True)

In [ ]:
df.info()

In [ ]:
# Verify that each file has a unique annotation and a corresponding WAV file
wavs = {p.stem: p for p in sorted(DATA_DIR.glob("*.wav"))}
assert df[["file", "transcription"]].notna().all().all()
assert df["file"].is_unique and set(df["file"]) == set(
    wavs
), "Annotations and WAV files differ"

In [ ]:
# Add audio paths and file prefixes for later use
df["audio_path"] = df["file"].map(wavs)
df["file_prefix"] = df["file"].str.extract(r"^([^_]+)", expand=False)

In [ ]:
# Summarize the number of samples per source TSV file
df.groupby("source_tsv").size().rename("samples").to_frame()

## Parsing, normalization, and synthetic checks


In [ ]:
def parse_response(payload: dict) -> str:
    """
    Parse the ASR API response to extract the transcribed text.

    Args:
        payload (dict): The JSON response from the ASR API,
            expected to contain a "document" key with a list of segments.

    Raises:
        ValueError: If the response does not contain a valid document list or if any segment is not a dictionary.

    Returns:
        str: The concatenated transcribed text from the document segments, with leading and trailing whitespace removed.
    """
    if not isinstance(payload, dict) or not isinstance(payload.get("document"), list):
        raise ValueError("Response must contain a document list")

    if any(not isinstance(x, dict) for x in payload["document"]):
        raise ValueError("Document segments must be objects")

    return " ".join(
        x.get("text", "").strip()
        for x in payload["document"]
        if x.get("text", "").strip()
    )


def normalize(text: str) -> str:
    """
    Normalize the input text by removing accents, converting to lowercase,
    and replacing non-alphanumeric characters with spaces.

    Args:
        text (str): The input text to normalize.

    Returns:
        str: The normalized text.
    """
    text = unicodedata.normalize("NFKD", str(text)).lower()
    text = "".join(c for c in text if not unicodedata.combining(c))
    return " ".join(
        "".join(c if c.isalnum() or c.isspace() else " " for c in text).split()
    )


assert (
    parse_response({"document": [{"text": " Hola "}, {"text": "mundo"}]})
    == "Hola mundo"
)

## Parallel transcription with retries and atomic checkpoints
Successful rows are skipped when rerun; failed rows are attempted again.


In [ ]:
_local = threading.local()


def session():
    if not hasattr(_local, "session"):
        _local.session = requests.Session()
    return _local.session


def transcribe_one(fid: str, path: str) -> dict:
    """
    Transcribe a single audio file using the ASR API, with retry logic and error handling.

    Args:
        fid (str): The file identifier.
        path (str): The path to the audio file.

    Returns:
        dict: A dictionary containing the transcription results and metadata.
    """
    started = time.monotonic()
    error = ""

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            with path.open("rb") as stream:
                response = session().post(
                    API_TRANSCRIBE_ENDPOINT,
                    files={"file": (path.name, stream, "audio/wav")},
                    data={"response_format": "diarized_json"},
                    timeout=TIMEOUT,
                )
            response.raise_for_status()
            payload = response.json()
            return dict(
                file=fid,
                hypothesis=parse_response(payload),
                response_json=json.dumps(payload, ensure_ascii=False),
                status="success",
                error="",
                attempts=attempt,
                elapsed_seconds=time.monotonic() - started,
            )
        except (requests.RequestException, ValueError) as exc:
            error = f"{type(exc).__name__}: {exc}"
            if isinstance(exc, ValueError) or attempt > MAX_RETRIES:
                break
            time.sleep(2 ** (attempt - 1))

    return dict(
        file=fid,
        hypothesis="",
        response_json="",
        status="error",
        error=error,
        attempts=attempt,
        elapsed_seconds=time.monotonic() - started,
    )


def load_checkpoint() -> pd.DataFrame:
    """
    Load the transcription results from a checkpoint CSV file if it exists,

    Returns:
        pd.DataFrame: A DataFrame containing the transcription results,
            or an empty DataFrame with predefined columns if the checkpoint file does not exist.
    """
    return (
        pd.read_csv(CHECKPOINT, keep_default_na=False)
        if CHECKPOINT.exists()
        else pd.DataFrame(columns=RESULT_COLUMNS)
    )


def save_checkpoint(records: dict) -> None:
    """
    Save the transcription results to a checkpoint CSV file.

    Args:
        records (dict): A dictionary of transcription results, where keys are file identifiers
            and values are dictionaries containing the transcription data.
    """
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    temp = CHECKPOINT.with_suffix(".tmp")
    pd.DataFrame(sorted(records.values(), key=lambda x: x["file"])).to_csv(
        temp, index=False, columns=RESULT_COLUMNS
    )
    temp.replace(CHECKPOINT)


def transcribe_dataset(data: pd.DataFrame) -> pd.DataFrame:
    """
    Transcribe a dataset of audio files using the ASR API,
    with checkpointing to allow resuming.

    Args:
        data (pd.DataFrame): A DataFrame containing the dataset to transcribe,
            expected to have "file" and "audio_path" columns.

    Returns:
        pd.DataFrame: A DataFrame containing the transcription results for the dataset.
    """
    records = {r["file"]: r for r in load_checkpoint().to_dict("records")}
    done = {k for k, v in records.items() if v["status"] == "success"}
    work = data.loc[~data.file.isin(done), ["file", "audio_path"]]
    work = work if SAMPLE_LIMIT is None else work.head(SAMPLE_LIMIT)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [
            pool.submit(transcribe_one, r.file, r.audio_path)
            for r in work.itertuples(index=False)
        ]
        for n, future in enumerate(tqdm(as_completed(futures), total=len(futures)), 1):
            result = future.result()
            records[result["file"]] = result
            if n % 25 == 0:
                save_checkpoint(records)

    if len(work):
        save_checkpoint(records)

    return load_checkpoint()


results = transcribe_dataset(df)

## Raw and normalized metrics


In [ ]:
df = df.merge(results, on="file", how="left")
df["status"] = df.status.fillna("pending")
df["hypothesis"] = df.hypothesis.fillna("")
df["reference_normalized"] = df.transcription.map(normalize)
df["hypothesis_normalized"] = df.hypothesis.map(normalize)
evaluation = df[df.status == "success"].copy()

In [ ]:
def edit_counts(reference: str, hypothesis: str, unit: str) -> tuple[int, int, float]:
    """
    Evaluate the edit counts and error rate between a reference and hypothesis transcription
    using the jiwer library, based on the specified unit of measurement.

    Args:
        reference (str): The reference transcription.
        hypothesis (str): The hypothesis transcription.
        unit (str): The unit of measurement, either "word" or "character".

    Returns:
        tuple[int, int, float]: A tuple containing the number of errors, the total number of units, and the error rate.
    """
    out = (jiwer.process_words if unit == "word" else jiwer.process_characters)(
        reference, hypothesis
    )
    units = out.hits + out.substitutions + out.deletions
    errors = out.substitutions + out.deletions + out.insertions
    return errors, units, errors / units if units else np.nan


for label, ref, hyp in [
    ("raw", "transcription", "hypothesis"),
    ("normalized", "reference_normalized", "hypothesis_normalized"),
]:
    for unit, metric in [("word", "wer"), ("character", "cer")]:
        values = [
            edit_counts(r, h, unit) for r, h in zip(evaluation[ref], evaluation[hyp])
        ]
        for i, name in enumerate(["errors", "units", "rate"]):
            evaluation[f"{label}_{metric}_{name}"] = [v[i] for v in values]


def aggregate(group: pd.DataFrame) -> pd.Series:
    """
    Aggregate the edit counts and error rates for a group of samples,
    calculating the total number of samples and the overall error rates for both raw and normalized transcriptions.

    Args:
        group (pd.DataFrame): A DataFrame group containing the samples to aggregate.

    Returns:
        pd.Series: A Series containing the total number of samples
            and the overall error rates for raw and normalized transcriptions.
    """
    result = {"samples": len(group)}
    for label in ["raw", "normalized"]:
        for metric in ["wer", "cer"]:
            units = group[f"{label}_{metric}_units"].sum()
            result[f"{label}_{metric}"] = (
                group[f"{label}_{metric}_errors"].sum() / units if units else np.nan
            )
    return pd.Series(result)


assert edit_counts("hola mundo", "hola gente", "word")[2] == 0.5

display(
    pd.Series(
        {
            "total": len(df),
            "success": len(evaluation),
            "failed": (df.status == "error").sum(),
            "pending": (df.status == "pending").sum(),
            "empty_hypotheses": evaluation.hypothesis.str.strip().eq("").sum(),
        }
    ).to_frame("value")
)
display(aggregate(evaluation).to_frame("value"))

## Summaries, breakdowns, and audio playback


In [ ]:
metric_cols = [
    "raw_wer_rate",
    "raw_cer_rate",
    "normalized_wer_rate",
    "normalized_cer_rate",
]

print("Error rate percentiles:")
display(
    evaluation[metric_cols + ["elapsed_seconds"]].describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print("Error rates by source TSV:")
display(evaluation.groupby("source_tsv").apply(aggregate, include_groups=False))

print("Worst transcriptions by normalized WER:")
worst = evaluation.sort_values("normalized_wer_rate", ascending=False).head(20)
display(
    worst[
        ["file", "transcription", "hypothesis", "raw_wer_rate", "normalized_wer_rate"]
    ]
)
if df["status"].eq("error").any():
    print("Errors in transcriptions:")
    display(df[df["status"] == "error"][["file", "attempts", "error"]].head(50))

In [ ]:
def display_sample(index: int) -> None:
    """
    Display a sample from the evaluation dataframe.

    Args:
        index (int): The index of the sample to display.
    """
    row = evaluation.loc[index]
    display(
        pd.Series(
            {
                "file": row.file,
                "reference": row.transcription,
                "hypothesis": row.hypothesis,
                "raw WER": row.raw_wer_rate,
                "normalized WER": row.normalized_wer_rate,
            }
        ).to_frame("value")
    )
    display(Audio(filename=str(row.audio_path)))

In [ ]:
# Display a random sample from the evaluation results
if not evaluation.empty:
    display_sample(evaluation.sample(1).index[0])

In [ ]:
# Assess the worst transcriptions by normalized WER
for index in worst.index:
    display_sample(index)
    print("-" * 40)